# Discrete Visualization

*Scenario:* You have been appointed as a data scientist at the National Travel and Tourism Office within the International Trade Administration Industry and Analysis. This national bureau is dedicated to promoting tourism in the United States, and your role involves contributing to the International Visitation and Spending in the United States project. As the fiscal year comes to a close, you have received a request from the headquarters to generate insights using the provided tourist visitation numbers for different states in the U.S. Your specific task is to create a notebook with visualizations that can interact with three years' worth of U.S. international visitation data.

## Data Preparation

In [1]:
import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def load_data() -> pd.DataFrame:
    """This function should load the data as described in the assignment description"""
    
    # Read Excel files, remove header rows
    
    df_2017 = pd.read_excel('assets/US_States_Visited_2017.xlsx', skiprows = range(6))
    df_2018 = pd.read_excel('assets/US_States_Visited_2018.xlsx', skiprows = range(7))
    df_2019 = pd.read_excel('assets/US_States_Visited_2019.xlsx', skiprows = range(6))

    # Remove footers
    
    df_2017 = df_2017[:-13]
    df_2018 = df_2018[:-7]
    df_2019 = df_2019[:-8]
    
    # Rename columns
    
    df_2017.columns = ['Rank', 'state', '2016 Market Share', 'visitation_2016', 
                       '2017 Market Share', 'visitation_2017', 'Volume % Change']
    df_2018.columns = ['Rank', 'state', '2018 Market Share', 'visitation_2018', 
                       'Volume % Change', '2017 Market Share', 'visitation_2017']
    df_2019.columns = ['Rank', 'state', '2019 Market Share', 'visitation_2019', 
                       'Volume % Change', '2018 Market Share', 'visitation_2018']
    
    # Multiply x1000
    
    df_2017['visitation_2016'] = df_2017['visitation_2016'].multiply(1000)
    df_2017['visitation_2017'] = df_2017['visitation_2017'].multiply(1000)
    df_2018['visitation_2018'] = df_2018['visitation_2018'].multiply(1000)
    df_2018['visitation_2017'] = df_2018['visitation_2017'].multiply(1000)
    df_2019['visitation_2019'] = df_2019['visitation_2019'].multiply(1000)
    df_2019['visitation_2018'] = df_2019['visitation_2018'].multiply(1000)

    # Strip white space
    
    df_2017['state'] = df_2017['state'].str.replace(' ', '')
    df_2018['state'] = df_2018['state'].str.replace(' ', '')
    df_2019['state'] = df_2019['state'].str.replace(' ', '')

    # Filter relevant columns
    
    df_2017 = df_2017[['state', 'visitation_2016', 'visitation_2017']]
    df_2018 = df_2018[['state', 'visitation_2018']]
    df_2019 = df_2019[['state', 'visitation_2019']]
    
    # Merge df_2017, df_2018, df_2019
    merged_US_states_visitation = df_2017.merge(df_2018, on='state', how='outer').merge(df_2019, on='state', 
                                                                                        how='outer')
    
    # Order alphabetically
    
    merged_US_states_visitation = merged_US_states_visitation.sort_values(by='state')
    merged_US_states_visitation = merged_US_states_visitation.reset_index(drop=True)
    
    return merged_US_states_visitation

## Bar Chart

In [2]:
def make_bar_chart(data):
    
    years = ['2016', '2017', '2018', '2019']
    
    for i, year in enumerate(years):
        visitationData = pd.to_numeric(data[f'visitation_{year}'], errors='coerce')
        
        #Bar chart
        
        plt.figure(figsize = (8, 4))
        
        plt.bar(data['state'], visitationData)
        plt.title(f'Visitation data {year}')
        
        # Visitation highest and lowest
                
        visitation_max = visitationData.max()
        visitation_min = visitationData.min()        
        max_state = data.loc[visitationData.idxmax(), 'state']
        min_state = data.loc[visitationData.idxmin(), 'state']
                
        # Creating markers for bars
                
        plt.plot(max_state, visitation_max, marker='v', color='green', markersize=15)
        plt.plot(min_state, visitation_min, 'x', color='red', markersize=15)
        plt.ylabel('Visitation (million)')
        plt.xlabel('State')
        plt.xticks(rotation=90)
        plt.show()


make_bar_chart(load_data())

FileNotFoundError: [Errno 2] No such file or directory: 'assets/US_States_Visited_2017.xlsx'

## Logarithmic Transformation

I chose to generate a logarithmic transformation of the plots based on what I learned from Professor Dowell's "Scaling and Transforming Data" lecture. The example from the lecture included Professor Dowell plotting the Dow Jones and using a logarithmic transformation to better visualize a skewed distribution. As the plots generated from question one are highly skewed, a logarithmic transformation allows us to reduce skewness and emphasize all features and facts of the data. Using log values reduces the difference between high ranging values and lower ones, while still maintaining their difference. Sorting the states allows the audience to discern which state has the highest visitation and which state has the lowest.

In [ ]:
def make_transformed_bar_chart(data):
    """This function applies a logarithmic transformation to the plots generated from Question 1"""

    years = ['2016', '2017', '2018', '2019']
    for i, year in enumerate(years):
        visitationData = pd.to_numeric(data[f'visitation_{year}'],errors='coerce')
        log_visitationData = np.log1p(visitationData)
        
        # Sorting data from lowest ---> highest
        
        visitationDataSorted = log_visitationData.sort_values().index
        
        transformed_visitationData = log_visitationData.loc[visitationDataSorted]
        
        states_sorted = data['state'].loc[visitationDataSorted]
                             
        # Generate bar chart
                             
        plt.figure(figsize=(8,4))
        plt.bar(states_sorted, transformed_visitationData)
        plt.title(f'Transformed Visitation data {year}')     
        plt.ylabel('Visitation (log)')
        plt.xlabel('State')
        plt.xticks(rotation=90)
        plt.show()

make_transformed_bar_chart(load_data())